# 📸 Урок 16 — Проект «Классификатор моих фото» (материалы преподавателя)

Рабочий шаблон: свои фото → аугментация → transfer learning → Gradio.

> ★ Ученики снимают 2–3 категории по 15–20 фото, раскладывают по папкам.

## Шаг 0 · Как загрузить фото
Структура папок (одна папка = один класс):
```
data/
  ручка/   pic1.jpg pic2.jpg ...
  чашка/   ...
  телефон/ ...
```
Загрузить папку `data` можно через панель «Файлы» слева или заархивировать и распаковать в Colab.

## Шаг 1 · Данные + аугментация

In [ ]:
import tensorflow as tf
from tensorflow import keras
train = keras.utils.image_dataset_from_directory('data', image_size=(160,160), batch_size=16)
class_names = train.class_names
print('Категории:', class_names)
augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
])

## Шаг 2 · Transfer learning на своих фото

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights='imagenet')
base.trainable = False
model = keras.Sequential([
    augment,
    keras.layers.Rescaling(1./127.5, offset=-1),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(len(class_names), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, epochs=8)

## Шаг 3 · Веб-интерфейс Gradio (публичная ссылка)

In [ ]:
!pip install gradio -q
import gradio as gr, tensorflow as tf
def predict(img):
    x = tf.image.resize(img, (160,160))[None, ...]
    p = model.predict(x)[0]
    return {class_names[i]: float(p[i]) for i in range(len(class_names))}
gr.Interface(fn=predict, inputs=gr.Image(), outputs=gr.Label(num_top_classes=3),
             title='Классификатор моих фото').launch(share=True)

---
**Итог.** Из немногих своих фото + аугментация + transfer learning получается рабочий классификатор с публичной ссылкой.